# Hospital ER Analysis — SQL Layer
**Tool: SQLite + Jupyter Notebook (via pandas / sqlite3)**

This notebook answers the following business questions:
- **QA1** — Satisfaction survey response-rate & non-response bias
- **QB3** — Departments with the worst wait-time / satisfaction combination
- **QD2** — Day/hour staffing gap (demand vs. service-level miss rate)
- **QD3** — Case Manager assignment vs. outcomes

In [2]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

conn = sqlite3.connect('hospital_er.db')
pd.read_sql("SELECT COUNT(*) AS total_visits FROM er_visits", conn)


,total_visits
0,9216


## QA1. Satisfaction survey response rate & non-response bias

Only a fraction of visits carry a satisfaction score. We check the response rate, then compare average wait time and admission rate **between responders and non-responders** — if they differ meaningfully, the dashboard's headline satisfaction average is built on a biased sample.

In [3]:
response_rate = pd.read_sql('''
    SELECT
        SUM("Has Satisfaction Score") AS responders,
        COUNT(*) - SUM("Has Satisfaction Score") AS non_responders,
        ROUND(100.0 * SUM("Has Satisfaction Score") / COUNT(*), 1) AS response_rate_pct
    FROM er_visits
''', conn)
response_rate


,responders,non_responders,response_rate_pct
0,2517,6699,27.3


In [4]:
bias_check = pd.read_sql('''
    SELECT
        CASE WHEN "Has Satisfaction Score" = 1 THEN 'Responded' ELSE 'No response' END AS survey_group,
        COUNT(*) AS visits,
        ROUND(AVG("Patient Waittime"), 1) AS avg_wait_time,
        ROUND(100.0 * SUM(CASE WHEN "Admission Status" = 'Admitted' THEN 1 ELSE 0 END) / COUNT(*), 1) AS admission_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN "Referred Flag" = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS referred_rate_pct
    FROM er_visits
    GROUP BY survey_group
''', conn)
bias_check


,survey_group,visits,avg_wait_time,admission_rate_pct,referred_rate_pct
0,No response,6699,35.2,50.4,40.9
1,Responded,2517,35.4,49.1,42.8


**Insight:** the survey captures only ~27% of visits (2,517 of 9,216). Average wait time is nearly identical between responders and non-responders (35.4 vs. 35.2 min) and admission/referral rates are within a few points of each other — so on these fields there's no strong sign of non-response bias. That's checked formally with a t-test in the Python notebook (QA1 continued). The response rate itself is still the headline finding: with only 1 in 4 visits represented, the dashboard's satisfaction average should be reported with a confidence interval and an explicit response-rate caveat, not as a single clean number.

## QB3. Which department referrals combine the longest wait and lowest satisfaction?

This ranks each department by patient volume, average wait time, and average satisfaction (among responders) to identify where operational attention is most needed.

In [5]:
dept_ranking = pd.read_sql('''
    SELECT
        "Department Referral" AS department,
        COUNT(*) AS visits,
        ROUND(AVG("Patient Waittime"), 1) AS avg_wait_time,
        ROUND(AVG(CASE WHEN "Has Satisfaction Score" = 1 THEN "Patient Satisfaction Score" END), 2) AS avg_satisfaction,
        ROUND(100.0 * SUM(CASE WHEN "Admission Status" = 'Admitted' THEN 1 ELSE 0 END) / COUNT(*), 1) AS admission_rate_pct
    FROM er_visits
    WHERE "Department Referral" <> 'No Referral'
    GROUP BY department
    ORDER BY avg_wait_time DESC
''', conn)
dept_ranking


,department,visits,avg_wait_time,avg_satisfaction,admission_rate_pct
0,Neurology,193,36.8,5.28,50.3
1,Physiotherapy,276,36.6,4.99,49.6
2,Gastroenterology,178,35.8,5.80,50.0
3,Cardiology,248,35.4,5.14,49.2
4,Orthopedics,995,35.0,4.86,50.1
5,General Practice,1840,34.9,5.06,48.2
6,Renal,86,34.7,4.57,53.5


**Insight:** departments with both above-average wait time *and* below-average satisfaction are the clearest candidates for added staffing or a fast-track pathway. `None` (no referral / treated in the ER directly) is excluded here since it isn't a specialty department, but is included in the notebook's full appendix query for completeness.

## QD2. Where does demand most exceed service capacity? (day × hour staffing gap)

For every day-of-week / 2-hour window, we calculate visit volume, average wait time, and the % of patients **missing** the 30-minute target. High volume + high miss-rate together flag the windows most in need of additional staffing.

In [6]:
staffing_gap = pd.read_sql('''
    SELECT
        "Admission DayName" AS day_of_week,
        "Hour Bucket" AS hour_window,
        COUNT(*) AS visits,
        ROUND(AVG("Patient Waittime"), 1) AS avg_wait_time,
        ROUND(100.0 * SUM(CASE WHEN "Waittime Status" = 'Target Missed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_target_missed
    FROM er_visits
    GROUP BY day_of_week, hour_window
    ORDER BY pct_target_missed DESC, visits DESC
    LIMIT 15
''', conn)
staffing_gap


,day_of_week,hour_window,visits,avg_wait_time,pct_target_missed
0,Thursday,22-24,116,37.2,70.7
1,Saturday,22-24,127,39.0,69.3
2,Monday,02-04,111,38.5,68.5
3,Tuesday,04-06,122,37.0,68.0
4,Wednesday,22-24,107,36.7,67.3
5,Tuesday,12-14,107,37.7,66.4
6,Monday,06-08,84,36.3,65.5
7,Tuesday,10-12,106,36.8,65.1
8,Thursday,10-12,111,37.4,64.9
9,Sunday,20-22,121,36.9,64.5


**Insight:** this table is the actionable version of the dashboard's day/hour volume heatmap — instead of just *where patients show up*, it shows *where they show up **and** the 30-minute target is being missed most*, which is the direct signal for where to schedule the next shift's extra staff.

## QD3. Does Case Manager assignment correlate with better outcomes?

Roughly half of visits have a Case Manager (CM) assigned. We compare average wait time, satisfaction, and admission rate between the two groups.

In [7]:
cm_outcomes = pd.read_sql('''
    SELECT
        "Case Manager Assigned" AS cm_status,
        COUNT(*) AS visits,
        ROUND(AVG("Patient Waittime"), 1) AS avg_wait_time,
        ROUND(AVG(CASE WHEN "Has Satisfaction Score" = 1 THEN "Patient Satisfaction Score" END), 2) AS avg_satisfaction,
        ROUND(100.0 * SUM(CASE WHEN "Admission Status" = 'Admitted' THEN 1 ELSE 0 END) / COUNT(*), 1) AS admission_rate_pct
    FROM er_visits
    GROUP BY cm_status
''', conn)
cm_outcomes


,cm_status,visits,avg_wait_time,avg_satisfaction,admission_rate_pct
0,CM Assigned,480,34.4,5.16,47.7
1,No CM,8736,35.3,4.98,50.2


**Insight:** only ~5% of visits (480 of 9,216) have a Case Manager assigned. CM-assigned visits show a modestly lower average wait time (34.4 vs. 35.3 min) but the satisfaction and admission-rate differences are small — on this data, CM assignment doesn't show a strong, clear outcome advantage, which is worth escalating: either coverage is too thin to move the aggregate numbers, or assignment isn't targeted at the visits that would benefit most. A formal t-test on this comparison is run in the Python notebook.

---
### Next: `Python_Statistical_and_ML_Analysis.ipynb` for significance testing, predictive models, segmentation, and forecasting.

In [8]:
conn.close()